In [1]:
import pathlib
import os
import datetime

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

import torch
from torch.utils.tensorboard.writer import SummaryWriter
import trimesh

import network, utils, dataset, train, visualization

%load_ext autoreload
%autoreload 2

/home/nikola/miniconda3/envs/adlr/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
# Model name
current_time = datetime.datetime.now().strftime("%b%d_%H-%M")
# denoiser_name = "Resnet_2_256"
denoiser_name = "3Objects_Resnet_2_256_200epochs_32batch"
experiment_name = f"{current_time}_{denoiser_name}"


config = {
    'experiment_name': experiment_name,
    'device': 'cuda:0',
    'is_overfit': True,
    'batch_size': 32,
    'resume_ckpt': None,
    'learning_rate': 0.0004,
    'step_size': 10, # scheduler step, one step is one batch
    'gamma': 1,
    'max_epochs': 200,
    'timesteps': 1000,
    'print_every_n': 2, # every n batches
    # 'validate_every_n': 25,
    # 'print_EMD_every_n': 1
}

model_config = {
    'denoiser_class': 'Denoiser',
    'last_epoch': 0,
    # 'hidden_dim': 512,
    # 'time_embedding_dim': 512
}

In [3]:
# declare device

device = torch.device('cpu')
if torch.cuda.is_available() and config['device'].startswith('cuda'):
    device = torch.device(config['device'])
    print('Using device:', config['device'])
else:
    print('Using CPU')

# create dataloaders
trainset = dataset.Dataset('train' if not config['is_overfit'] else 'overfit', config['timesteps'])
trainloader = torch.utils.data.DataLoader(trainset, batch_size=config['batch_size'], shuffle=True, num_workers=0, pin_memory=False)
print(len(trainloader))
# valset = dataset.Dataset('val' if not config['is_overfit'] else 'overfit', config['timesteps'])
# valloader = torch.utils.data.DataLoader(valset, batch_size=config['batch_size'], shuffle=False, num_workers=0)

denoiser = network.Denoiser()
diffuser = network.Diffuser(config['timesteps'])

# load model if resuming from checkpoint
# if config['resume_ckpt'] is not None:
#         utils.reload_model(denoiser, diffuser, config['experiment_name'], device)

# move model to specified device
denoiser.to(device)
diffuser.to(device)
optimizer = torch.optim.Adam(denoiser.parameters(), lr=config['learning_rate'])
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, config['step_size'], config['gamma'])
# scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.0006, steps_per_epoch=len(trainloader), epochs=config['max_epochs'], anneal_strategy='cos', three_phase=True)


#Run this code in terminal to start tensorboard: tensorboard --logdir=diffusion/nikola/logs/diffusion_training


total, trainable = utils.count_parameters(denoiser)
print(f"Total: {total:,} | Trainable: {trainable:,} | Model size: {utils.model_memory_size(denoiser):.3f} MB")

Using device: cuda:0
32
Total: 824,067 | Trainable: 824,067 | Model size: 3.144 MB


In [11]:
# start training
#tensorboard --logdir=diffusion/nikola/logs/diffusion2obj
# config['max_epochs'] = 200
# Create tensorboard writer    
log_path = pathlib.Path(f"logs/diffusion2obj/{datetime.datetime.now().strftime('%b%d')}/{config['experiment_name']}")
writer = SummaryWriter(log_path)

with open("debug.txt", "w", encoding="utf-8") as debug_file:
    train.train(denoiser, diffuser, trainloader, None, device, optimizer, scheduler, config, model_config, writer, debug_file)

writer.close()


[00/001] train_loss: 0.942
[00/003] train_loss: 0.706
[00/005] train_loss: 0.478
[00/007] train_loss: 0.320
[00/009] train_loss: 0.237
[00/011] train_loss: 0.213
[00/013] train_loss: 0.159
[00/015] train_loss: 0.209
[00/017] train_loss: 0.187
[00/019] train_loss: 0.150
[00/021] train_loss: 0.157
[00/023] train_loss: 0.170
[00/025] train_loss: 0.150
[00/027] train_loss: 0.132
[00/029] train_loss: 0.166
[00/031] train_loss: 0.184
Best Model:   0.28500055195763707
Error: No TensorBoard log files discovered in: logs/diffusion2obj/May21_12-46_3Objects_Resnet_2_256_200epochs_32batch
[01/001] train_loss: 0.211
[01/003] train_loss: 0.191
[01/005] train_loss: 0.152
[01/007] train_loss: 0.203
[01/009] train_loss: 0.102
[01/011] train_loss: 0.114
[01/013] train_loss: 0.147
[01/015] train_loss: 0.172
[01/017] train_loss: 0.139
[01/019] train_loss: 0.143
[01/021] train_loss: 0.127
[01/023] train_loss: 0.196
[01/025] train_loss: 0.120
[01/027] train_loss: 0.130
[01/029] train_loss: 0.155
[01/031] tr

In [22]:
#Load a model:
experiment_name = "May20_23-27_Resnet_2_256_180epochs_32batch"
utils.reload_model(denoiser, diffuser, None, None, experiment_name, 'checkpoint', device)

({'experiment_name': 'May20_23-27_Resnet_2_256_80epochs_32batch',
  'device': 'cuda:0',
  'is_overfit': True,
  'batch_size': 32,
  'resume_ckpt': None,
  'learning_rate': 0.0004,
  'step_size': 10,
  'gamma': 1,
  'max_epochs': 200,
  'timesteps': 1000,
  'print_every_n': 2},
 {'denoiser_class': 'Denoiser', 'last_epoch': 171})

In [29]:
generateDDIM = 0
generateDDPM = 1
number_of_points = 2048
DDIM_steps = 40
number_of_DDIM_iterations = 20
save = False
start = 20

if generateDDIM:
    for i in range(start, start + number_of_DDIM_iterations):
        generated_pc_ddim = network.sample_ddim(denoiser, diffuser, n_points=number_of_points, steps=DDIM_steps)
        generated_pcd_ddim = trimesh.PointCloud(generated_pc_ddim.squeeze().cpu().numpy())
        utils.visualize_comparison(trainset[1], generated_pc_ddim, window_name="DDIM Target (Red) vs Generated (Blue)")
        if save:
            path = pathlib.Path(f"output/{experiment_name}")
            path.mkdir(parents=True, exist_ok=True)
            generated_pcd_ddim.export(path / f"ddim{DDIM_steps}_{number_of_points}_{i}.obj")

if generateDDPM:
    generated_pc_ddpm = network.sample_ddpm(denoiser, diffuser, n_points=number_of_points)
    generated_pcd_ddpm = trimesh.PointCloud(generated_pc_ddpm.squeeze().cpu().numpy())
    utils.visualize_comparison(trainset[0], generated_pc_ddpm, window_name="DDPM Target (Red) vs Generated (Blue)")

Visualizing: Target is RED, Generated is BLUE.


In [10]:

number_of_points = 2048
generated_pc, samples_list = network.sample_and_capture(denoiser, diffuser, n_points=number_of_points, save_every=10)
utils.visualize_diffusion_progress(samples_list, window_name="Diffusion Process")

In [14]:
# Optionally, save the generated point clouds to disk
path = pathlib.Path(f"output/{experiment_name}")
path.mkdir(parents=True, exist_ok=True)
generated_pcd_ddim.export(path / f"ddim{DDIM_steps}_{number_of_points}.obj")
# generated_pcd_ddpm.export(f"output/{experiment_name}_ddpm.obj")